# Nhận diện object: YOLOv8x-oiv7 + Segment Anything (tuỳ chọn)

- **Đầu vào:** `<keyframe-dir>/<video_id>/<shot_id>/<keyframe>.jpg`.
- **Đầu ra:** `/kaggle/working/od/<video_id>/<shot_id>/<keyframe>.json`
  `{video_id, shot_id, keyframe_id, detections: [{model, class, confidence, box, mask_polygon}]}`.

Lưu ý: `build_object_index.py` đọc file object của BTC (`objects/<video_id>/<n>.json` với
`detection_class_entities` / `detection_scores`), không đọc định dạng này. Bật GPU.

In [ ]:
!pip install -q ultralytics opencv-python-headless
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

## Cấu hình
Độ phân giải cao, TTA và ngưỡng thấp để không bỏ sót vật thể; SAM tinh chỉnh các lớp có hình
dạng bất thường thành polygon.

In [ ]:
from pathlib import Path

KEYFRAME_DIR = Path("/kaggle/input/datasets/trncqun/aic26-keyframes/keyframes")
OUTPUT_DIR = Path("/kaggle/working/od")
YOLO_WEIGHTS = "yolov8x-oiv7.pt"  # tự tải lần đầu
SAM_WEIGHTS = Path("/kaggle/input/sam-weights/sam_vit_h_4b8939.pth")  # None để bỏ SAM
CONF = 0.15
IMGSZ = 1280
TTA = True
IRREGULAR_CLASSES = ("Animal", "Fish", "Tool", "Organ", "Coral")

SHARD_INDEX = 1  # ← đổi theo máy
NUM_SHARDS = 1

## Model

In [ ]:
import json

import cv2
import numpy as np
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO


def select_shard(items, shard_index, num_shards, weight=None):
    """Part shard_index (1-based) of num_shards. Items are sorted first so every machine agrees.

    Without weight: contiguous chunks. With weight: the heaviest item goes to the lightest
    shard (e.g. by video size), so all machines finish at about the same time.
    """
    assert 1 <= shard_index <= num_shards, "SHARD_INDEX phải nằm trong [1, NUM_SHARDS]"
    ordered = sorted(items)
    if num_shards == 1:
        return ordered
    if weight is None:
        size = -(-len(ordered) // num_shards)
        return ordered[(shard_index - 1) * size : shard_index * size]
    shards = [[] for _ in range(num_shards)]
    totals = [0.0] * num_shards
    for item in sorted(ordered, key=weight, reverse=True):
        lightest = totals.index(min(totals))
        shards[lightest].append(item)
        totals[lightest] += weight(item)
    return sorted(shards[shard_index - 1])


def matches_any(video_id, selectors):
    """True when no selector is given or video_id equals / starts with one of them."""
    return not selectors or any(video_id == s or video_id.startswith(s) for s in selectors)


device = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLO(YOLO_WEIGHTS)
sam_predictor = None
if SAM_WEIGHTS is not None:
    from segment_anything import SamPredictor, sam_model_registry

    sam = sam_model_registry["vit_h"](checkpoint=str(SAM_WEIGHTS))
    sam.to(device=device)
    sam_predictor = SamPredictor(sam)


def sharpen(image):
    """Unsharp mask: 1.5 x original - 0.5 x blurred, to help the detector on soft frames."""
    return cv2.addWeighted(image, 1.5, cv2.GaussianBlur(image, (9, 9), 10.0), -0.5, 0)


def mask_polygon(box):
    masks, _, _ = sam_predictor.predict(
        point_coords=None, point_labels=None, box=np.array(box)[None, :], multimask_output=False
    )
    contours, _ = cv2.findContours(
        masks[0].astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    approx = cv2.approxPolyDP(largest, 0.005 * cv2.arcLength(largest, True), True)
    return approx.squeeze().tolist() if approx.ndim > 2 else approx.tolist()

## Chạy nhận diện
Chạy lại cell để tiếp tục: ảnh đã có JSON được bỏ qua.

In [ ]:
images = select_shard(list(KEYFRAME_DIR.rglob("*.jpg")), SHARD_INDEX, NUM_SHARDS)
print(f"Shard {SHARD_INDEX}/{NUM_SHARDS}: {len(images)} ảnh, SAM: {sam_predictor is not None}")
unreadable = []
for image_path in tqdm(images, desc="Object detection"):
    relative = image_path.relative_to(KEYFRAME_DIR)
    out_json = (OUTPUT_DIR / relative).with_suffix(".json")
    if out_json.exists():
        continue
    image = cv2.imread(str(image_path))
    if image is None:
        unreadable.append(image_path)
        continue
    image = sharpen(image)
    results = model.predict(image, verbose=False, conf=CONF, imgsz=IMGSZ, augment=TTA)
    if sam_predictor is not None:
        sam_predictor.set_image(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    detections = []
    for result in results:
        for box in result.boxes:
            name = model.names[int(box.cls[0])]
            xyxy = box.xyxy[0].cpu().numpy().tolist()
            irregular = any(c in name for c in IRREGULAR_CLASSES)
            detections.append(
                {
                    "model": Path(YOLO_WEIGHTS).stem,
                    "class": name,
                    "confidence": round(float(box.conf[0]), 4),
                    "box": xyxy,
                    "mask_polygon": mask_polygon(xyxy) if sam_predictor and irregular else None,
                }
            )
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(
        json.dumps(
            {
                "video_id": relative.parts[0],
                "shot_id": relative.parts[1] if len(relative.parts) > 2 else "",
                "keyframe_id": image_path.stem,
                "detections": detections,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
print(f"Xong; {len(unreadable)} ảnh không đọc được", unreadable[:5])

## Nén kết quả
Tải file nén ở tab **Output** của Kaggle rồi giải nén vào `data/od/` trên máy build.

In [ ]:
import shutil

archive = shutil.make_archive(
    str(OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_shard{SHARD_INDEX}of{NUM_SHARDS}"),
    "zip",
    root_dir=str(OUTPUT_DIR.parent),
    base_dir=OUTPUT_DIR.name,
)
print(f"{archive}: {Path(archive).stat().st_size / 2**20:.1f} MB")